# Week 5 – Spark DataFrame Assignment
**Objective:** Understand Spark fundamentals and perform data cleaning, transformation, and aggregation using DataFrames.

In [1]:
# ─────────────────────────────────────────────────────────────
# SETUP – Start a local SparkSession and load the dataset
# ─────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

spark = SparkSession.builder \
    .appName("Week5Assignment") \
    .master("local[*]") \
    .getOrCreate()

# Load dataset (update path if needed)
df = spark.read.csv("week5_dataset.csv", header=True, inferSchema=True)
df.show()
df.printSchema()

Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: org.apache.spark.SparkException: Invalid Spark URL: spark://HeartbeatReceiver@Krish_Kalra:53188
	at org.apache.spark.rpc.RpcEndpointAddress$.apply(RpcEndpointAddress.scala:66)
	at org.apache.spark.rpc.netty.NettyRpcEnv.asyncSetupEndpointRefByURI(NettyRpcEnv.scala:143)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.executor.Executor.<init>(Executor.scala:489)
	at org.apache.spark.scheduler.local.LocalEndpoint.<init>(LocalSchedulerBackend.scala:66)
	at org.apache.spark.scheduler.local.LocalSchedulerBackend.start(LocalSchedulerBackend.scala:136)
	at org.apache.spark.scheduler.TaskSchedulerImpl.start(TaskSchedulerImpl.scala:229)
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:629)
	at org.apa

---
## Q1 – Limitations of MapReduce vs. Spark

In [ ]:
# Q1 ANSWER:
# MapReduce has several key limitations that make Spark the preferred choice:
#
# 1. Disk I/O Bottleneck:
#    MapReduce writes intermediate results to HDFS after every Map and Reduce step.
#    This creates heavy disk I/O. Spark keeps intermediate data in memory (RAM),
#    making it 10x–100x faster for most workloads.
#
# 2. No Support for Iterative Algorithms:
#    ML algorithms (e.g., gradient descent) repeat the same computation many times.
#    MapReduce reads data from disk on every iteration. Spark caches the data in
#    memory once and reuses it across iterations — a massive speedup.
#
# 3. Only Batch Processing:
#    MapReduce supports only batch jobs. Spark supports batch, streaming
#    (Spark Streaming / Structured Streaming), interactive SQL, and ML.
#
# 4. High Latency:
#    Starting a MapReduce job involves JVM startup and data shuffling through disk.
#    Spark minimises latency through in-memory execution and a DAG scheduler.
#
# 5. Complex Programming Model:
#    MapReduce forces you to express everything as Map → Shuffle → Reduce.
#    Spark provides high-level APIs (DataFrames, SQL, MLlib) that are far more expressive.
#
# 6. No Native Streaming or Graph Support:
#    MapReduce needs add-ons like Storm for streaming. Spark bundles
#    Structured Streaming, GraphX, and MLlib out of the box.

print("Q1: Conceptual – no code output required.")

---
## Q2 – In-Memory Computing for Iterative ML

In [ ]:
# Q2 ANSWER:
# HOW SPARK USES IN-MEMORY COMPUTING:
#
# Spark stores RDDs / DataFrames in RAM using .cache() or .persist().
# Once the data is loaded into memory, every subsequent iteration
# reads directly from RAM instead of going to disk.
#
# Example – Logistic Regression in MLlib runs 100 iterations:
#   MapReduce: reads full dataset from HDFS 100 times  → 100 × disk read
#   Spark    : reads dataset once, caches in memory    → 1  × disk read
#
# KEY MECHANISM – Lineage Graph (DAG):
#   Spark builds a Directed Acyclic Graph of transformations.
#   Cached data lets the engine skip re-computation on repeated passes.
#   If a partition is lost, Spark re-computes ONLY that partition using lineage.
#
# RESULT:
#   Benchmark (Apache): Logistic Regression on 100 GB data
#     Hadoop MapReduce  →  110 seconds/iteration
#     Spark (cached)    →    1.5 seconds/iteration  (~73× faster)

# Code illustration – cache before an iterative operation
df.cache()   # pins the DataFrame in memory after first action
print("DataFrame cached. Subsequent actions skip disk I/O.")

---
## Q3 – Remove Duplicate Rows on Specific Columns

In [ ]:
# Q3 ANSWER:
# .dropDuplicates([col_list]) keeps the FIRST occurrence of each
# unique combination of the specified columns and discards the rest.

df_no_duplicates = df.dropDuplicates(["user_id", "transaction_date"])

print(f"Rows BEFORE dedup: {df.count()}")
print(f"Rows AFTER  dedup: {df_no_duplicates.count()}")
df_no_duplicates.show()

---
## Q4 – Filter by Region Then Group by Category

In [ ]:
# Q4 ANSWER:
# Step 1 – filter() keeps only rows where region == 'West'.
# Step 2 – groupBy() groups remaining rows by product_category.
# Step 3 – agg(avg()) computes the average sale_amount per category.

df_sales = df   # using our loaded DataFrame as df_sales

result_q4 = (
    df_sales
    .filter(F.col("region") == "West")
    .groupBy("product_category")
    .agg(F.avg("sale_amount").alias("avg_sale_amount"))
)

result_q4.show()

---
## Q5 – .na.drop() vs .na.fill()

In [ ]:
# Q5 ANSWER:
# .na.drop() → REMOVES rows that contain null values.
#   Use when a row with missing data is useless to your analysis.
#   Example: df.na.drop()          – drops any row with at least 1 null
#            df.na.drop(subset=["email"]) – drops rows where email is null
#
# .na.fill() → REPLACES null values with a specified substitute.
#   Use when you want to keep the row but fix the missing value.
#   Example: df.na.fill("Unknown") – fills ALL string nulls with 'Unknown'
#            df.na.fill({"price": 0}) – fills nulls in price column with 0

# Fill null values in the 'status' column with the string 'Unknown'
df_filled = df.na.fill({"status": "Unknown"})

print("Before fill – status column nulls:")
df.filter(F.col("status").isNull()).select("user_id", "status").show()

print("After fill – status column:")
df_filled.select("user_id", "status").show()

---
## Q6 – Count Records per City (Count > 100)

In [ ]:
# Q6 ANSWER:
# groupBy("city")        – groups rows by city
# count()                – counts records in each group
# .filter(count > 100)   – post-aggregation filter using HAVING semantics
#
# Note: You cannot use a WHERE clause on an aggregated column; the filter
# must come AFTER the aggregation (equivalent to SQL HAVING clause).

result_q6 = (
    df
    .groupBy("city")
    .agg(F.count("*").alias("record_count"))
    .filter(F.col("record_count") > 100)
)

result_q6.show()
# (Our sample dataset has < 100 rows per city, so result will be empty –
#  the logic is correct and would work on a larger real-world dataset.)

---
## Q7 – Immutability of Spark DataFrames

In [ ]:
# Q7 ANSWER:
# Spark DataFrames are IMMUTABLE – once created, you cannot modify them in place.
# Every transformation (drop, rename, filter, etc.) returns a NEW DataFrame.
# The original DataFrame is never altered.
#
# HOW THIS AFFECTS DATA CLEANING:
#
# 1. You must always ASSIGN the result of a transformation to a new variable
#    (or re-assign to the same variable).
#    WRONG:  df.drop("raw_timestamp")          # original df unchanged!
#    RIGHT:  df = df.drop("raw_timestamp")     # reassign to overwrite
#
# 2. This enables LINEAGE TRACKING – Spark records every step as a DAG.
#    If a node fails, Spark re-computes from the last checkpoint using lineage.
#
# 3. It makes pipelines reproducible – the original data always exists
#    and intermediate steps can be inspected independently.

# Example – drop a column and rename another (each returns a NEW DataFrame)
df_cleaned = df.drop("raw_timestamp")               # drop column
df_cleaned = df_cleaned.withColumnRenamed("status", "user_status")  # rename

print("Original columns:", df.columns)
print("Cleaned  columns:", df_cleaned.columns)

---
## Q8 – Filter by Age Range and Subscription Type

In [ ]:
# Q8 ANSWER:
# .filter() (alias .where()) applies row-level conditions.
# Multiple conditions are combined with & (AND) or | (OR).
# Parentheses are REQUIRED around each condition when using & or |.

result_q8 = df.filter(
    (F.col("age").between(18, 30)) &          # inclusive age range 18–30
    (F.col("subscription") == "Premium")       # subscription is Premium
)

result_q8.select("user_id", "age", "subscription", "city").show()

---
## Q9 – Why Handle Nulls Before Aggregations?

In [ ]:
# Q9 ANSWER:
# Spark's built-in aggregation functions (sum, avg, min, max) SILENTLY SKIP
# null values. While this avoids crashes, it can produce misleading results:
#
# Problem 1 – INCORRECT AVERAGES:
#   If 3 out of 10 prices are null, avg() divides by 7 (not 10).
#   The average is inflated/deflated depending on what the nulls represent.
#
# Problem 2 – INCORRECT TOTALS:
#   sum() ignores nulls, so the total revenue is understated if
#   some transactions have a null price.
#
# Problem 3 – SILENT ERRORS:
#   No exception is raised – the wrong number looks like the right number.
#   These bugs are very hard to catch after the fact.
#
# BEST PRACTICE:
#   Before aggregating, always either:
#     a) Fill nulls with a sensible default  → df.na.fill({"price": 0})
#     b) Drop rows where the key column is null → df.na.drop(subset=["price"])

print("Price nulls:", df.filter(F.col("price").isNull()).count())
print("avg(price) with nulls   :", df.agg(F.avg("price")).collect()[0][0])
print("avg(price) after filling:", df.na.fill({"price": 0}).agg(F.avg("price")).collect()[0][0])

---
## Q10 – Cast and Rename a Timestamp Column

In [ ]:
# Q10 ANSWER:
# .cast(TimestampType()) converts the column's data type from StringType
# to TimestampType so Spark can perform time-based operations on it.
# .withColumnRenamed() renames the column to a cleaner name.

df_with_ts = (
    df
    .withColumn("raw_timestamp", F.col("raw_timestamp").cast(TimestampType()))
    .withColumnRenamed("raw_timestamp", "event_time")
)

df_with_ts.select("user_id", "event_time").show()
df_with_ts.printSchema()

---
## Q11 – The Shuffle Process in Wide Transformations

In [ ]:
# Q11 ANSWER:
# WHAT IS A SHUFFLE?
# A shuffle is the process of redistributing data across the cluster's
# partitions so that rows with the same grouping key end up on the same
# executor (worker node).
#
# STEP-BY-STEP during groupBy("city").count():
#   Stage 1 (Map):    Each executor reads its local partition and computes
#                     a partial count per city.
#   Shuffle (Write):  Each executor writes shuffle files to local disk,
#                     partitioned by the hash of the grouping key.
#   Shuffle (Read):   Executors fetch the relevant partitions from all
#                     other executors across the network.
#   Stage 2 (Reduce): Each executor merges the partial counts for its
#                     assigned keys to produce the final result.
#
# WHY IS IT A WIDE TRANSFORMATION?
# A NARROW transformation (e.g., filter, select) processes each partition
# independently – no data moves between partitions.
# A WIDE transformation (e.g., groupBy, join, distinct) requires data
# from MULTIPLE partitions to compute a single output record.
# Because data must cross partition/executor boundaries, it requires a
# shuffle and creates a new stage boundary in the DAG.
#
# COST:
#   Shuffles involve disk I/O and network transfer – they are the most
#   expensive operations in a Spark job. Minimise them by:
#     - Filtering early (reduce data before shuffling)
#     - Using broadcast joins for small tables
#     - Persisting/caching data before repeated wide operations

print("Q11: Conceptual – no code output required.")

---
## Q12 – Remove Rows with Null Email OR Empty Username

In [ ]:
# Q12 ANSWER:
# We want to KEEP rows where BOTH conditions are clean, so we filter
# to rows where email is NOT null AND username is NOT an empty string.
#
# Equivalent: drop rows where email IS null OR username IS empty string.

df_clean_contacts = df.filter(
    F.col("email").isNotNull() &             # remove rows with null email
    (F.col("username") != "") &              # remove rows with empty username
    F.col("username").isNotNull()            # also guard against null username
)

print(f"Rows before cleaning: {df.count()}")
print(f"Rows after  cleaning: {df_clean_contacts.count()}")
df_clean_contacts.select("user_id", "email", "username").show()

---
## Q13 – Multiple Statistics with .agg()

In [ ]:
# Q13 ANSWER:
# .agg() accepts multiple aggregation functions in a single call,
# each passed as a separate argument.
# Use .alias() to give each output column a meaningful name.

result_q13 = df.agg(
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.avg("price").alias("mean_price")
)

result_q13.show()

# You can also use it after groupBy to compute stats per group:
df.groupBy("product_category").agg(
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.avg("price").alias("mean_price")
).show()

---
## Q14 – Risk of inferSchema=True with Messy Dates

In [ ]:
# Q14 ANSWER:
# inferSchema=True tells Spark to scan the data and GUESS the data type
# of each column automatically.
#
# RISKS WITH MESSY OR INCONSISTENT DATE FORMATS:
#
# 1. INCORRECT TYPE INFERENCE:
#    If some dates are '2025-01-15' and others are '15/01/2025',
#    Spark may infer the column as StringType instead of DateType,
#    or fail to parse them at all.
#
# 2. NULL INJECTION:
#    Rows whose date format doesn't match the inferred format are
#    silently converted to null (with spark.sql.legacy.timeParserPolicy=LEGACY)
#    or throw a runtime exception (with CORRECTED mode).
#    Either way, you LOSE or CORRUPT data without a clear warning.
#
# 3. INCONSISTENCY ACROSS PARTITIONS:
#    inferSchema samples only a subset of rows (default 1,000 rows).
#    If the first 1,000 rows all use one format but later rows use another,
#    the inferred schema is wrong for part of the data.
#
# BEST PRACTICE:
#    Define the schema explicitly using StructType / StructField and
#    use F.to_timestamp() / F.to_date() with an explicit format string:
#      F.to_timestamp(F.col("date_col"), "yyyy-MM-dd HH:mm:ss")

print("Q14: Conceptual – no code output required.")

---
## Q15 – Full Data Processing Pipeline

In [ ]:
# Q15 ANSWER:
# A complete pipeline that:
#   Step 1 – Removes duplicate rows (based on user_id + transaction_date)
#   Step 2 – Fills null prices with 0
#   Step 3 – Groups by store_id and calculates total revenue (sum of price)

pipeline_result = (
    df
    # Step 1: Remove duplicates
    .dropDuplicates(["user_id", "transaction_date"])

    # Step 2: Fill null prices with 0
    .na.fill({"price": 0})

    # Step 3: Group by store and compute total revenue
    .groupBy("store_id")
    .agg(F.sum("price").alias("total_revenue"))
    .orderBy("store_id")
)

print("=== Final Pipeline Result: Total Revenue per Store ===")
pipeline_result.show()

# ─────────────────────────────────────────────────────────────
# INSIGHT:
# Running the three steps in this order matters:
#  - Deduplication first ensures we don't double-count the same
#    transaction when summing revenue.
#  - Filling nulls before aggregation ensures null prices contribute
#    0 to the total instead of being silently ignored.
#  - GroupBy last consolidates the clean data into the final metric.
# ─────────────────────────────────────────────────────────────